# Affinage — deux modules entraînés sur SciFact

Le zéro-shot plafonne à **35,3** de F1 au niveau abstract, sous le zéro-shot de
2020 (38,4). VeriSci atteint 48,5 parce qu'il est **entraîné** sur les 809
affirmations de `train`. C'est ce qu'on fait ici.

## Architecture, celle de Wadden et al. 2020

**Sélecteur de justifications** — note chaque phrase d'un abstract face à
l'affirmation. Entrée `[phrase, SEP, affirmation]`, sortie binaire. Négatifs :
les abstracts cités étiquetés NOINFO, plus les phrases non-justificatives des
abstracts SUPPORT et CONTRADICT.

**Classifieur d'étiquette** — lit l'affirmation et les phrases retenues, rend
SUPPORT, CONTRADICT ou NOINFO. Entrée `[phrases retenues, SEP, affirmation]`.

**Un avantage que VeriSci n'avait pas** : le classifieur part des poids d'un
modèle NLI dont la tête produit déjà *entailment / neutral / contradiction* —
soit exactement SUPPORT / NOINFO / CONTRADICT. On adapte au domaine au lieu de
réapprendre la tâche.

## Points de contrôle, par module — table 3 du papier, jeu dev

| Module | Repère |
|---|---|
| Sélection de justifications, RoBERTa-large entraîné sur SciFact | P 73,7 · R 70,5 · **F1 72,1** |
| Sélection, SciBERT | P 74,5 · R 74,3 · **F1 74,4** |
| Prédiction d'étiquette, RoBERTa-large | **exactitude 75,7** |
| Prédiction d'étiquette, SciBERT | exactitude 69,2 |

Chaque module est mesuré seul, avec les abstracts d'or, **avant** d'être enchaîné.
Si un module dévie de son repère, on le sait avant que l'erreur se propage.

## Cible finale — table 7, jeu dev, régime ouvert

| Système | phrase | abstract |
|---|---|---|
| Zéro-shot 2020 | 28,4 | 38,4 |
| ce système en zéro-shot | 26,6 | 35,3 |
| **VeriSci** | **42,6** | **48,5** |
| VeriSci, abstracts fournis | 60,6 | 72,5 |
| plafond de notre récupération | — | 89,7 |

## 1. Environnement

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "AUCUN GPU")

In [ ]:
!pip -q install transformers torch nltk pandas 2>&1 | tail -2
import torch, transformers, numpy as np, json, os, random
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())
GRAINE = 0
random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)

## 2. Données et évaluateur officiel

In [ ]:
import tarfile, zipfile, urllib.request

if not os.path.exists("scifact"):
    urllib.request.urlretrieve(
        "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip",
        "scifact.zip")
    zipfile.ZipFile("scifact.zip").extractall(".")
if not os.path.exists("data/claims_dev.jsonl"):
    urllib.request.urlretrieve(
        "https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz", "v.tar.gz")
    tarfile.open("v.tar.gz").extractall(".")

os.makedirs("evaluate/lib", exist_ok=True)
B = "https://raw.githubusercontent.com/allenai/scifact/master/verisci/evaluate"
urllib.request.urlretrieve(f"{B}/pipeline.py", "evaluate/pipeline.py")
for f in ["__init__.py", "data.py", "metrics.py"]:
    urllib.request.urlretrieve(f"{B}/lib/{f}", f"evaluate/lib/{f}")

corpus = {str(json.loads(l)["doc_id"]): json.loads(l)
          for l in open("data/corpus.jsonl", encoding="utf-8")}
train_all = [json.loads(l) for l in open("data/claims_train.jsonl", encoding="utf-8")]
dev = [json.loads(l) for l in open("data/claims_dev.jsonl", encoding="utf-8")]

# Le seuil se regle sur une part de train tenue a l'ecart, jamais sur dev.
random.Random(GRAINE).shuffle(train_all)
n_val = int(0.15 * len(train_all))
val, train = train_all[:n_val], train_all[n_val:]
print(f"train {len(train)} | val {len(val)} (réglage du seuil) | dev {len(dev)} (rapport)")
assert len(corpus) == 5183 and len(dev) == 300

## 3. Construire les données du sélecteur de justifications

Négatifs, exactement comme le papier : « les abstracts cités étiquetés NOINFO,
ainsi que les phrases non-justificatives des abstracts SUPPORTS et REFUTES ».

In [ ]:
def donnees_selection(claims):
    X, y = [], []
    for c in claims:
        ev = c.get("evidence") or {}
        for doc in c.get("cited_doc_ids", []):
            doc = str(doc)
            if doc not in corpus:
                continue
            phrases = corpus[doc]["abstract"]
            ors = {s for g in ev.get(doc, []) for s in g["sentences"]}
            for i, ph in enumerate(phrases):
                X.append((ph, c["claim"]))
                y.append(1 if i in ors else 0)
    return X, np.array(y)

Xtr, ytr = donnees_selection(train)
Xva, yva = donnees_selection(val)
print(f"sélection — entraînement {len(Xtr)} phrases, dont {ytr.sum()} justificatives "
      f"({100*ytr.mean():.1f} %)")
print(f"            validation   {len(Xva)} phrases, dont {yva.sum()} justificatives")

## 4. Entraîner le sélecteur de justifications

In [ ]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODELE_SEL = "microsoft/deberta-v3-base"
tok_sel = AutoTokenizer.from_pretrained(MODELE_SEL)
sel = AutoModelForSequenceClassification.from_pretrained(MODELE_SEL, num_labels=2).to("cuda")

class Paires(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], int(self.y[i])

def assembler(tok, lot, longueur=256):
    X = [b[0] for b in lot]; y = torch.tensor([b[1] for b in lot])
    e = tok([a for a, _ in X], [b for _, b in X], padding=True, truncation=True,
            max_length=longueur, return_tensors="pt")
    return e, y

def entrainer(modele, tok, X, y, epoques=3, lot=32, lr=2e-5, poids=None):
    dl = DataLoader(Paires(X, y), batch_size=lot, shuffle=True,
                    collate_fn=lambda b: assembler(tok, b))
    opt = torch.optim.AdamW(modele.parameters(), lr=lr, weight_decay=0.01)
    total = len(dl) * epoques
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    perte = torch.nn.CrossEntropyLoss(weight=poids.to("cuda") if poids is not None else None)
    scaler = torch.cuda.amp.GradScaler()
    modele.train()
    for ep in range(epoques):
        cumul = 0.0
        for n, (e, cible) in enumerate(dl):
            e = {k: v.to("cuda") for k, v in e.items()}; cible = cible.to("cuda")
            opt.zero_grad()
            with torch.cuda.amp.autocast():
                p = perte(modele(**e).logits, cible)
            scaler.scale(p).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            cumul += p.item()
            if (n + 1) % 100 == 0:
                print(f"  époque {ep+1}  lot {n+1}/{len(dl)}  perte {cumul/(n+1):.4f}")
        print(f"époque {ep+1} terminée — perte moyenne {cumul/len(dl):.4f}")
    modele.eval()
    return modele

# Les justificatives sont minoritaires : on repondere pour ne pas apprendre
# a toujours repondre non.
poids = torch.tensor([1.0, float((ytr == 0).sum() / max((ytr == 1).sum(), 1))])
print(f"poids de la classe positive : {poids[1]:.2f}")
sel = entrainer(sel, tok_sel, Xtr, ytr, epoques=3, lot=32, poids=poids)

## 5. Point de contrôle du sélecteur

Mesuré sur `dev`, **abstracts d'or fournis** — c'est le régime de la table 3.

Repères : RoBERTa-large entraîné sur SciFact **F1 72,1**, SciBERT **F1 74,4**.

In [ ]:
@torch.no_grad()
def noter(modele, tok, X, lot=128, longueur=256):
    sorties = []
    for i in range(0, len(X), lot):
        p = X[i:i + lot]
        e = tok([a for a, _ in p], [b for _, b in p], padding=True, truncation=True,
                max_length=longueur, return_tensors="pt").to("cuda")
        with torch.cuda.amp.autocast():
            sorties.append(torch.softmax(modele(**e).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(sorties) if sorties else np.zeros(0)

Xdev_or, ydev_or = donnees_selection(dev)
scores_dev = noter(sel, tok_sel, Xdev_or)

# seuil regle sur val, jamais sur dev
scores_val = noter(sel, tok_sel, Xva)
def prf(scores, y, t):
    p = scores >= t
    vp = int((p & (y == 1)).sum()); fp = int((p & (y == 0)).sum()); fn = int((~p & (y == 1)).sum())
    P = vp / (vp + fp) if vp + fp else 0.0
    R = vp / (vp + fn) if vp + fn else 0.0
    return P, R, 2 * P * R / (P + R) if P + R else 0.0

meilleur = max(((t, prf(scores_val, yva, t)[2]) for t in np.arange(0.05, 0.96, 0.05)),
               key=lambda x: x[1])
SEUIL_SEL = float(meilleur[0])
P, R, F = prf(scores_dev, ydev_or, SEUIL_SEL)
print(f"seuil réglé sur val : {SEUIL_SEL:.2f}")
print(f"\nsélection sur dev, abstracts d'or : P {100*P:.1f}  R {100*R:.1f}  F1 {100*F:.1f}")
print(f"  repère RoBERTa-large (papier)   : P 73.7  R 70.5  F1 72.1")
print(f"  repère SciBERT (papier)         : P 74.5  R 74.3  F1 74.4")

## 6. Classifieur d'étiquette, initialisé depuis les poids NLI

La tête du modèle NLI produit déjà *entailment / neutral / contradiction*. On
ne remplace pas cette tête : on la réutilise en associant SUPPORT à
`entailment`, NOINFO à `neutral`, CONTRADICT à `contradiction`.

In [ ]:
MODELE_ETIQ = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
tok_etiq = AutoTokenizer.from_pretrained(MODELE_ETIQ)
etiq = AutoModelForSequenceClassification.from_pretrained(MODELE_ETIQ).to("cuda")

id2label = {int(k): v.lower() for k, v in etiq.config.id2label.items()}
I_POUR   = next(i for i, v in id2label.items() if v.startswith("entail"))
I_NEUTRE = next(i for i, v in id2label.items() if v.startswith("neutral"))
I_CONTRE = next(i for i, v in id2label.items() if v.startswith("contradic"))
VERS_INDICE = {"SUPPORT": I_POUR, "NOINFO": I_NEUTRE, "CONTRADICT": I_CONTRE}
VERS_NOM = {v: k for k, v in VERS_INDICE.items()}
print("association :", {k: id2label[v] for k, v in VERS_INDICE.items()})

def donnees_etiquette(claims, phrases_par=None):
    """phrases_par[(id, doc)] = indices des phrases ; None = justifications d'or."""
    X, y = [], []
    for c in claims:
        ev = c.get("evidence") or {}
        for doc in c.get("cited_doc_ids", []):
            doc = str(doc)
            if doc not in corpus:
                continue
            groupes = ev.get(doc)
            if phrases_par is None:
                idx = sorted({s for g in groupes for s in g["sentences"]}) if groupes else []
            else:
                idx = phrases_par.get((c["id"], doc), [])
            if not idx:
                if groupes:
                    continue          # preuve sans phrase : cas inexploitable
                idx = list(range(min(3, len(corpus[doc]["abstract"]))))
            texte = " ".join(corpus[doc]["abstract"][i] for i in idx)
            X.append((texte, c["claim"]))
            y.append(VERS_INDICE[groupes[0]["label"] if groupes else "NOINFO"])
    return X, np.array(y)

Xe_tr, ye_tr = donnees_etiquette(train)
Xe_va, ye_va = donnees_etiquette(val)
from collections import Counter
print(f"étiquette — entraînement {len(Xe_tr)} exemples : "
      f"{ {VERS_NOM[k]: v for k, v in Counter(ye_tr.tolist()).items()} }")
etiq = entrainer(etiq, tok_etiq, Xe_tr, ye_tr, epoques=3, lot=8, lr=1e-5)

## 7. Point de contrôle du classifieur

Mesuré sur `dev` avec les **justifications d'or** en entrée — régime de la
table 3. Repères : RoBERTa-large **75,7** d'exactitude, SciBERT **69,2**.

In [ ]:
@torch.no_grad()
def classer_etiq(X, lot=32):
    out = []
    for i in range(0, len(X), lot):
        p = X[i:i + lot]
        e = tok_etiq([a for a, _ in p], [b for _, b in p], padding=True, truncation=True,
                     max_length=384, return_tensors="pt").to("cuda")
        with torch.cuda.amp.autocast():
            out.append(etiq(**e).logits.float().cpu().numpy())
    return np.vstack(out) if out else np.zeros((0, 3))

Xe_dev, ye_dev = donnees_etiquette(dev)
pred = classer_etiq(Xe_dev).argmax(1)
exactitude = float((pred == ye_dev).mean())
print(f"exactitude sur dev, justifications d'or : {100*exactitude:.1f}")
print(f"  repère RoBERTa-large (papier)         : 75.7")
print(f"  repère SciBERT (papier)               : 69.2")

print()
for nom, i in VERS_INDICE.items():
    m = ye_dev == i
    if m.sum():
        print(f"  {nom:<11} {int(m.sum()):>4} exemples, exactitude {100*(pred[m] == i).mean():.1f}")

## 8. Chaîne complète sur `dev` — récupération, sélection, étiquetage

Régime ouvert : BM25 récupère, le sélecteur choisit les phrases, le classifieur
décide. Un abstract étiqueté NOINFO n'est pas émis, comme l'exige l'évaluateur.

In [ ]:
import re
from collections import Counter as Cnt
from nltk.stem.porter import PorterStemmer

ARRET = set('''a an and are as at be but by for if in into is it no not of on or
such that the their then there these they this to was will with'''.split())
_r, _cache = PorterStemmer(), {}
def normaliser(t):
    out = []
    for m in re.findall(r"[a-z0-9]+", t.lower()):
        if m in ARRET: continue
        s = _cache.get(m)
        if s is None: s = _r.stem(m); _cache[m] = s
        out.append(s)
    return out

class BM25:
    def __init__(self, docs, k1=0.9, b=0.4):
        self.k1, self.b = k1, b
        j = [normaliser(d) for d in docs]
        self.n = len(j)
        self.lg = np.array([len(d) for d in j], dtype=np.float32)
        self.moy = float(self.lg.mean())
        brut = {}
        for i, doc in enumerate(j):
            for t, f in Cnt(doc).items(): brut.setdefault(t, []).append((i, f))
        self.index = {}
        for t, post in brut.items():
            idx = np.array([p[0] for p in post], dtype=np.int32)
            frq = np.array([p[1] for p in post], dtype=np.float32)
            df = len(post)
            self.index[t] = (idx, frq, float(np.log(1 + (self.n - df + .5) / (df + .5))))
    def scores(self, q):
        s = np.zeros(self.n, dtype=np.float32)
        for t in normaliser(q):
            e = self.index.get(t)
            if e is None: continue
            idx, frq, idf = e
            norme = 1 - self.b + self.b * self.lg[idx] / self.moy
            s[idx] += idf * (frq * (self.k1 + 1)) / (frq + self.k1 * norme)
        return s

doc_ids = list(corpus)
lex = BM25([f"{corpus[d]['title']} {' '.join(corpus[d]['abstract'])}" for d in doc_ids])

K_ABSTRACTS, MAX_PHRASES = 3, 3

def chaine(claims):
    sortie = []
    for c in claims:
        s = lex.scores(c["claim"])
        top = np.argpartition(-s, K_ABSTRACTS)[:K_ABSTRACTS]
        docs = [doc_ids[i] for i in top[np.argsort(-s[top])]]

        paires, reperes = [], []
        for doc in docs:
            ph = corpus[doc]["abstract"]
            reperes.append((doc, len(paires), len(paires) + len(ph)))
            paires.extend((p, c["claim"]) for p in ph)
        notes = noter(sel, tok_sel, paires) if paires else np.zeros(0)

        preuve = {}
        for doc, a, b in reperes:
            z = notes[a:b]
            idx = np.where(z >= SEUIL_SEL)[0]
            if len(idx) == 0:
                continue
            idx = idx[np.argsort(-z[idx])][:MAX_PHRASES]
            idx = sorted(int(i) for i in idx)
            texte = " ".join(corpus[doc]["abstract"][i] for i in idx)
            lg = classer_etiq([(texte, c["claim"])])[0]
            nom = VERS_NOM[int(lg.argmax())]
            if nom != "NOINFO":
                preuve[str(doc)] = {"label": nom, "sentences": idx}
        sortie.append({"id": c["id"], "evidence": preuve})
    return sortie

def evaluer_officiel(predictions, gold):
    with open("pred.jsonl", "w") as f:
        for p in predictions: f.write(json.dumps(p) + "\n")
    r = subprocess.run(["python", "pipeline.py", "--gold", f"../{gold}",
                        "--corpus", "../data/corpus.jsonl", "--prediction", "../pred.jsonl",
                        "--output", "../m.json"], cwd="evaluate", capture_output=True, text=True)
    if not os.path.exists("m.json"):
        print(r.stdout[-1200:], r.stderr[-1200:]); raise RuntimeError("évaluateur en échec")
    m = json.load(open("m.json")); os.remove("m.json")
    return m

print("chaîne complète sur dev…")
m = evaluer_officiel(chaine(dev), "data/claims_dev.jsonl")

## 9. Résultat

In [ ]:
ph = m["sentence_label"]["f1"] * 100
ab = m["abstract_rationalized"]["f1"] * 100

REPERES = [
    ("Zéro-shot (FEVER), 2020",        28.4, 38.4),
    ("ce système, zéro-shot",          26.6, 35.3),
    ("VeriSci, régime ouvert",         42.6, 48.5),
    ("VeriSci, abstracts fournis",     60.6, 72.5),
    ("Justifications fournies",        79.9, 83.0),
]
print(f"{'système':<36}{'phrase':>10}{'abstract':>11}")
print("-" * 57)
for nom, p, a in REPERES:
    print(f"{nom:<36}{p:>10.1f}{a:>11.1f}")
print("-" * 57)
marque = "  ← dépasse VeriSci" if ab > 48.5 else ""
print(f"{'ce système, affiné':<36}{ph:>10.1f}{ab:>11.1f}{marque}")
print("-" * 57)
print(f"{'plafond de la récupération':<36}{'':>10}{89.7:>11.1f}")

print()
for cle, lib in [("sentence_selection", "phrase, sélection seule"),
                 ("sentence_label", "phrase, sélection + étiquette"),
                 ("abstract_label_only", "abstract, étiquette seule"),
                 ("abstract_rationalized", "abstract, étiquette + justification")]:
    d = m[cle]
    print(f"  {lib:<38} P {d['precision']*100:>5.1f}  R {d['recall']*100:>5.1f}  F1 {d['f1']*100:>5.1f}")

json.dump({"modele_selection": MODELE_SEL, "modele_etiquette": MODELE_ETIQ,
           "seuil_selection": SEUIL_SEL, "k_abstracts": K_ABSTRACTS,
           "metriques_dev": m,
           "controles": {"selection_F1_dev_or": 100*F, "etiquette_exactitude_dev_or": 100*exactitude},
           "reperes": {n: {"phrase": p, "abstract": a} for n, p, a in REPERES}},
          open("resultats_affinage.json", "w"), indent=2)
print("\nécrit : resultats_affinage.json")

## 10. Récupérer

In [ ]:
from google.colab import files
files.download("resultats_affinage.json")